# Fixed Effect Comparison: Sporulation Distance Analysis

This notebook performs a mixed-effects analysis to examine how spectral distance from baseline (0 DPI) changes over time (DPI) for the spectra.

## Analysis Pipeline
1. Load and average sporulation ROIs per Sample × DPI
2. Apply PCA transformation and compute Euclidean distance to baseline
3. Fit mixed-effects model: Distance ~ DPI + (1|Sample)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib import cm
from scipy import stats

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

import statsmodels.formula.api as smf

# Set matplotlib style for better-looking plots
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.size': 11,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'DejaVu Sans', 'Liberation Sans'],
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 16,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linewidth': 0.5,
})

## 1. Load and Average Sporulation ROIs

Function to load spatial sporulation data and average ROIs within each Sample × DPI combination.

In [ ]:
def load_spectra_avg(
    csv_path: str,
    tray_col: int = 1,
    sample_col: int = 2,
    dpi_col: int = 3,
    label_col: int = 4,
    spectra_start_col: int = 5,
    id_col: int = None,
    keyword: str = "sporulation",
):
    """
    Load CSV and average spectra ROIs per Sample × DPI.
    
    Parameters:
    -----------
    csv_path : str
        Path to CSV file
    sample_col : int
        Column index for Sample
    dpi_col : int
        Column index for DPI
    label_col : int
        Column index for label/ROI type
    spectra_start_col : int
        First column index containing spectral data
    keyword : str
        Keyword to filter spectra ROIs

    Returns:
    --------
    spor_avg : pd.DataFrame
        Averaged spectra data with Sample, DPI, and spectral columns
    spectral_cols : list[str]
        List of spectral column names
    """
    df = pd.read_csv(csv_path)
    
    if id_col is not None:
        sample = (df.iloc[:, tray_col].astype(str) + "_" + df.iloc[:, sample_col].astype(str)+ "_" + df.iloc[:, id_col].astype(str)).rename("Sample") 
    else:
        sample = (df.iloc[:, tray_col].astype(str) + "_" + df.iloc[:, sample_col].astype(str)).rename("Sample")
    dpi_raw = df.iloc[:, dpi_col].astype(str).str.strip()
    # More tolerant DPI extraction: match the first integer (could be "0", "01", "1", etc)
    dpi = dpi_raw.str.extract(r"(\d+)")[0]
    dpi_numeric = pd.to_numeric(dpi, errors="coerce")
    dpi_final = dpi_numeric.astype("Int64").rename("DPI")
    label = df.iloc[:, label_col].astype(str)
    spectra = df.iloc[:, spectra_start_col:].apply(pd.to_numeric, errors="coerce")

    # Fix dpi mask to use numeric comparison only between 0 and 6 inclusive
    dpi_valid = dpi_numeric.between(0, 6, inclusive="both")
    mask = label.str.contains(keyword, case=False, na=False) & dpi_valid

    df_output = pd.concat([sample, dpi_final, spectra], axis=1).loc[mask]
    # Average ROIs within each leaf x dpi
    spectra_avg = df_output.groupby(["Sample", "DPI"], as_index=False).mean(numeric_only=True)

    spectral_cols = [c for c in spectra_avg.columns if c not in ("Sample", "DPI")]
    return spectra_avg, spectral_cols

## 2. PCA Transformation and Distance to Baseline

Apply PCA to spectral data and compute Euclidean distance from each time point to the baseline (0 DPI) for each sample.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline


def pca_signed_distance_to_baseline(
    df_avg: pd.DataFrame,
    spectral_cols: list[str],
    baseline_dpi: int = 0,
    late_dpi: int = 6,          # Option A: disease axis = mean(late) - mean(baseline)
    variance_threshold: float = 0.95,
    sample_col: str = "Sample",
    dpi_col: str = "DPI",
    signed_col: str = "Distance",
):
    X = df_avg[spectral_cols].to_numpy(dtype=float)
    ids = df_avg[sample_col].astype(str).to_numpy()
    dpis = pd.to_numeric(df_avg[dpi_col], errors="coerce").astype(int).to_numpy()

    # --- PCA components chosen by variance threshold ---
    scaler = StandardScaler(with_mean=True, with_std=True)
    X_scaled = scaler.fit_transform(X)

    full_pca = PCA().fit(X_scaled)
    cumvar = np.cumsum(full_pca.explained_variance_ratio_)
    n_components_selected = int(np.searchsorted(cumvar, variance_threshold) + 1)
    print(f"Selected {n_components_selected} PCA components to reach {variance_threshold*100:.1f}% variance.")

    pipe = Pipeline(
        steps=[
            ("scaler", scaler),
            ("pca", PCA(n_components=n_components_selected, random_state=0)),
        ]
    )
    Z = pipe.fit_transform(X)

    if late_dpi is None:
        late_dpi = int(np.max(dpis))

    # --- Build disease direction (unit vector) using paired leaves that have both baseline and late DPI ---
    baseline_vecs, late_vecs = [], []
    for sid in np.unique(ids):
        idx0 = np.where((ids == sid) & (dpis == baseline_dpi))[0]
        idxL = np.where((ids == sid) & (dpis == late_dpi))[0]
        if len(idx0) == 0 or len(idxL) == 0:
            continue
        baseline_vecs.append(Z[idx0].mean(axis=0))
        late_vecs.append(Z[idxL].mean(axis=0))

    if len(baseline_vecs) == 0:
        raise ValueError(f"No samples have both baseline_dpi={baseline_dpi} and late_dpi={late_dpi}.")

    baseline_mean = np.mean(baseline_vecs, axis=0)
    late_mean = np.mean(late_vecs, axis=0)

    disease_dir = late_mean - baseline_mean
    norm = float(np.linalg.norm(disease_dir))
    if norm == 0.0:
        raise ValueError("Disease direction has zero norm (baseline_mean == late_mean). Choose a different late_dpi.")
    disease_dir = disease_dir / norm

    # --- Signed projection onto disease axis, baseline per leaf ---
    signed = np.full(len(df_avg), np.nan, dtype=float)
    for i, sid in enumerate(ids):
        idx0 = np.where((ids == sid) & (dpis == baseline_dpi))[0]
        if len(idx0) == 0:
            continue
        base = Z[idx0].mean(axis=0)
        signed[i] = float(np.dot(Z[i] - base, disease_dir))

    out = df_avg[[sample_col, dpi_col]].copy()
    out[dpi_col] = dpis
    out[signed_col] = signed
    out = out.dropna(subset=[signed_col]).reset_index(drop=True)

    return out

## 4. Visualization Functions

Create publication-quality visualizations of the mixed-effects model results with individual trajectories and model predictions.

In [ ]:
def fit_mixedlm_random_intercept(dist_df, sample_col="Sample"):
    """
    Fit mixed-effects model with random intercepts.
    
    Parameters:
    -----------
    dist_df : pd.DataFrame
        DataFrame with Sample, DPI, and Distance columns
    sample_col : str
        Column name for sample/group identifier
    
    Returns:
    --------
    dd : pd.DataFrame
        Cleaned dataframe
    res : statsmodels.regression.mixed_linear_model.MixedLMResults
        Fitted model results
    """
    dd = dist_df.copy()
    dd["DPI"] = pd.to_numeric(dd["DPI"], errors="coerce")
    dd["Distance"] = pd.to_numeric(dd["Distance"], errors="coerce")
    dd = dd.dropna(subset=["DPI", "Distance", sample_col])
    dd["DPI"] = dd["DPI"].astype(int)

    dd["DPIc"] = dd["DPI"] - dd["DPI"].mean()

    model = smf.mixedlm("Distance ~ DPIc", data=dd, groups=dd[sample_col])
    res = model.fit(reml=False, method="powell", maxiter=5000, disp=False)
    return dd, res

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf


def plot_violin_strip_with_mixedlm(
    dist_df,
    sample_col="Sample",
    jitter=0.10,
    point_size=18,
    violin_alpha=0.25,
):
    # ---- Fit mixed model ----
    dd = dist_df.copy()
    dd["DPI"] = pd.to_numeric(dd["DPI"], errors="coerce")
    dd["Distance"] = pd.to_numeric(dd["Distance"], errors="coerce")
    dd = dd.dropna(subset=["DPI", "Distance", sample_col])
    dd["DPI"] = dd["DPI"].astype(int)
    dd["DPIc"] = dd["DPI"] - dd["DPI"].mean()

    model = smf.mixedlm("Distance ~ DPIc", data=dd, groups=dd[sample_col])
    res = model.fit(reml=False, method="powell", maxiter=5000, disp=False)

    b0 = float(res.fe_params["Intercept"])
    b1 = float(res.fe_params["DPIc"])
    try:
        sd_intercept = float(np.sqrt(res.cov_re.iloc[0, 0]))
    except Exception:
        sd_intercept = 0.0

    # ---- Prepare axes ----
    dpis = np.sort(dd["DPI"].unique())
    dpi_mean = float(dd["DPI"].mean())

    x = dpis.astype(float)
    y_mean = b0 + b1 * (x - dpi_mean)
    y_lo = y_mean - sd_intercept
    y_hi = y_mean + sd_intercept

    fig, ax = plt.subplots(figsize=(10, 4))

    # ---- Violin (neutral color) ----
    data_by_dpi = [dd.loc[dd["DPI"] == d, "Distance"].to_numpy() for d in dpis]
    parts = ax.violinplot(
        data_by_dpi,
        positions=x,
        widths=0.70,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for body in parts["bodies"]:
        body.set_alpha(violin_alpha)
        body.set_facecolor("gray")

    # ---- Assign ONE color per leaf ----
    leaf_ids = dd[sample_col].unique()
    cmap = plt.get_cmap("tab20")  # good for up to ~20 leaves
    leaf_colors = {
        leaf: cmap(i % cmap.N) for i, leaf in enumerate(leaf_ids)
    }

    # ---- Strip plot (leaf-colored points) ----
    rng = np.random.default_rng(0)
    for leaf, g in dd.groupby(sample_col):
        color = leaf_colors[leaf]
        for d in dpis:
            y = g.loc[g["DPI"] == d, "Distance"].to_numpy()
            if len(y) == 0:
                continue
            xj = d + rng.uniform(-jitter, jitter, size=len(y))
            ax.scatter(xj, y, s=point_size, color=color, alpha=0.6)

    # ---- Mixed-model mean + random-intercept SD ----
    ax.plot(x, y_mean, linewidth=3.0, color="black")
    ax.fill_between(x, y_lo, y_hi, color="black", alpha=0.18)

    # ---- Labels ----
   # ---- Labels (bold) ----
    ax.set_xlabel("DPI", fontsize=12, fontweight="bold")
    ax.set_ylabel("Distance (from 0 DPI)", fontsize=12, fontweight="bold")

    # ---- Tick labels (bold) ----
    ax.tick_params(axis="both", which="major", labelsize=11)
    for tick in ax.get_xticklabels() + ax.get_yticklabels():
        tick.set_fontweight("bold")


    ax.text(
        0.02,
        0.98,
        f"intercept (β_0) = {b0:.3f}\nSlope (β_DPI) = {b1:.3f}\nSD(random intercept) = {sd_intercept:.3f}\n"
        f"N={len(dd)}, leaves={dd[sample_col].nunique()}",
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=10,
        bbox=dict(boxstyle="round", alpha=0.12),
    )

    plt.tight_layout()
    plt.savefig("pca_distance_mixedlm_plot.svg", format="svg", dpi=1000)
    plt.show()

    return res

## 3. Mixed-Effects Model

Fit a mixed-effects linear model to test the relationship between Distance and DPI, with random intercepts for each Sample.

In [ ]:
def mixedlm_distance_vs_dpi(dist_df: pd.DataFrame):
    """
    Fit mixed-effects model: Distance ~ DPI + (1|Sample)
    
    Parameters:
    -----------
    dist_df : pd.DataFrame
        DataFrame with Sample, DPI, and Distance columns
    
    Returns:
    --------
    res : statsmodels.regression.mixed_linear_model.MixedLMResults
        Fitted model results
    """
    dd = dist_df.copy()
    dd["DPI"] = dd["DPI"].astype(int)
    dd["DPIc"] = dd["DPI"] - dd["DPI"].mean()

    # Random intercept + random slope
    m_rs = smf.mixedlm(
        "Distance ~ DPIc",
        data=dd,
        groups=dd["Sample"],      # or SampleID
        re_formula="~DPIc"
    )

    res_rs = m_rs.fit(reml=False, method="powell", maxiter=8000, disp=False)
    print(res_rs.summary())
    
    return res_rs

## Main Analysis

Run the complete pipeline: load data, compute distances, and fit the mixed-effects model.

In [ ]:
csv_path = "../../data/Hyperbird/NewTimeCourse_filtered_shrunken.csv"
# csv_path = "../../data/Hyperbird/regions_spectra/all_regions_spectra.csv"

spectra_avg, spectral_cols = load_spectra_avg(
    csv_path=csv_path,
    tray_col=1,
    sample_col=2,
    dpi_col=3,
    label_col=4,
    spectra_start_col=5,
    keyword="DM",
)

print("spectra_avg shape:", spectra_avg.shape)
print("n spectral cols:", len(spectral_cols))

In [ ]:
dist_df = pca_signed_distance_to_baseline(
    df_avg=spectra_avg,
    spectral_cols=spectral_cols,
    baseline_dpi=0,
    late_dpi=6,
    variance_threshold=0.95,
)

print("Distance DF shape:", dist_df.shape)

In [ ]:
res = mixedlm_distance_vs_dpi(dist_df)

In [ ]:
res = plot_violin_strip_with_mixedlm(dist_df)